# SaaS billing benchmark at scale

Same generate → verify → score loop as notebooks 2–4, on a larger billing domain.

- Generate a balanced set of tasks (types × personalities), then keep only verified ones.
- Evaluate the multi-agent pipeline and compare models.

**Lessons learned**
- At scale, generation needs balanced coverage and eligible starting rows.
- Outcome scoring (final DB + required phrases) is not enough: inspect traces; compare stronger and weaker agents. 
- Review agent execution traces to identify whether failures are due to agent mistakes or limitations in the evaluation criteria. This helps determine if task or metric improvements are needed.
- Single-run success metrics like `pass@1` can be highly variable; measuring consistency over multiple independent runs (`pass^k`) provides a more robust evaluation of agent reliability, though at increased computational or credit cost.
- Be critical: do you need to change the evaluation, or is it correctly evaluating the expected actions and communication keywords.


In [ ]:
import json
from collections import Counter

from aieng.syn_data.synbench.display import (  # noqa: E402
    show_actions,
    show_benchmark_by_task_type,
    show_pipeline_messages,
    show_session,
    show_task,
)
from utils.utils import get_root_directory


ROOT = get_root_directory()
DOMAIN_PATH = ROOT / "domains" / "mock_saas_billing"
OUT_DIR = ROOT / "data" / "benchmarks" / "mock_saas_billing"
OUT_DIR.mkdir(parents=True, exist_ok=True)
BENCHMARK_RESULTS_DIR = OUT_DIR / "results"
BENCHMARK_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Implementation root:", ROOT)
print("Domain:", DOMAIN_PATH)
print("Output:", OUT_DIR)

Implementation root: /Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/agent_benchmark_generation
Domain: /Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/agent_benchmark_generation/domains/mock_saas_billing
Output: /Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/agent_benchmark_generation/data/benchmarks/mock_saas_billing


## Step 1 — Load and inspect the scaled domain

Validate the complete domain bundle, then report aggregate sizes instead of printing the full database.

In [2]:
from aieng.syn_data.synbench.domain.loader import load_domain, validate_domain


errors = validate_domain(DOMAIN_PATH)
assert errors == [], f"Domain validation failed: {errors}"
domain = load_domain(DOMAIN_PATH)

print("Domain:", domain.manifest.name)
print("\nTask types:", list(domain.task_types.keys()))
print("Seed tasks:", len(domain.seed_tasks))
for collection in ("accounts", "subscriptions", "invoices"):
    print(f"{collection}: {len(domain.db[collection])}")

sample_invoice = next(iter(domain.db["invoices"].values()))
print("\nSample invoice:\n", json.dumps(sample_invoice, indent=2))

Domain: mock_saas_billing

Task types: ['inquiry', 'void_invoice', 'refuse_void', 'update_billing_email', 'update_seats']
Seed tasks: 10
accounts: 48
subscriptions: 96
invoices: 192

Sample invoice:
 {
  "invoice_id": "inv_avery_open",
  "account_id": "acct_avery",
  "subscription_id": "sub_avery_1",
  "status": "open",
  "amount_cents": 2898,
  "currency": "USD",
  "due_date": "2026-01-15",
  "billing_address": "100 Market St, Suite 1",
  "line_items": [
    {
      "description": "Starter plan seats",
      "quantity": 3,
      "unit_cents": 966
    }
  ]
}


## Step 2 — Inspect and replay seed tasks

The domain has two hand-verified seeds per task type. Replay one oracle to demonstrate the target database state used for scoring.

In [ ]:
from aieng.syn_data.synbench.environment.core import replay_actions
from aieng.syn_data.synbench.environment.hashing import db_hash


for task_type in domain.task_types:
    ids = [task.id for task in domain.seed_tasks if task.task_type == task_type]
    print(f"{task_type}: {ids}")

seed = domain.seed_tasks[0]
target_env = replay_actions(domain, seed.evaluation_criteria.actions)
print("\nReplayed seed:", seed.id)
print("Target DB hash:", db_hash(target_env.db))

inquiry: ['seed_inquiry_avery', 'seed_inquiry_noah']
void_invoice: ['seed_void_avery', 'seed_void_maya']
refuse_void: ['seed_refuse_avery', 'seed_refuse_luis']
update_billing_email: ['seed_email_noah', 'seed_email_priya']
update_seats: ['seed_seats_avery', 'seed_seats_luis']

Replayed seed: seed_inquiry_avery
Target DB hash: d8a90747cd51adbf7105b338ad1d5d0cff9f87fec41f2a83033c67b77ed7e033


### Don't run this cell if you have already generated the tasks.
## Step 3 — Generate tasks at scale with balanced constraints

Hold the task type fixed while sampling a fresh eligible invoice/account for each personality. The retry loop avoids duplicate accounts within a task type and skips inactive subscriptions for seat changes. With the defaults below this requests 25 drafts: 5 task types × 5 personalities.

In [4]:
from aieng.syn_data.synbench.generation.generator import TrajectoryGenerator
from aieng.syn_data.synbench.generation.sampler import ConstraintSampler


TASK_TYPES = list(domain.task_types.keys())
PERSONALITIES = [style["name"] for style in domain.user_simulator["personality_styles"]]
MAX_SAMPLE_ATTEMPTS = 100

sampler = ConstraintSampler(domain, seed=42)
generator = TrajectoryGenerator(domain, seed=42)
drafts = []
generation_rows = []
# generate one task for each task type and personality --> 25 tasks
for task_type in TASK_TYPES:
    used_accounts = set()
    for personality in PERSONALITIES:
        for _ in range(MAX_SAMPLE_ATTEMPTS):
            constraints = sampler.sample(
                task_type=task_type,
                personality_name=personality,
            )
            account_id = constraints.entities["account_id"]
            subscription = constraints.entity_context.get("subscriptions", {})
            if account_id in used_accounts:
                continue
            if task_type == "update_seats" and subscription.get("status") != "active":
                continue
            break
        else:
            raise RuntimeError(
                f"Could not find a distinct eligible account for {task_type}/{personality}",
            )

        used_accounts.add(account_id)
        task_id = f"scale_{task_type}_{personality}"
        draft = generator.generate_one(task_id=task_id, constraints=constraints)
        drafts.append(draft)
        generation_rows.append(
            {
                "task_type": task_type,
                "account_id": account_id,
                "user_name": draft.user_scenario.user_name,
                "personality": personality,
                "invoice_id": constraints.primary_id,
            },
        )

print(f"Generated {len(drafts)} drafts")
for row in generation_rows:
    print(
        f"{row['task_type']:<22} {row['personality']:<14} {row['account_id']:<12} {row['invoice_id']}",
    )

Generated 25 drafts
inquiry                rushed         acct_041     inv_041_4
inquiry                anxious        acct_008     inv_008_1
inquiry                rule_breaker   acct_noah    inv_noah_3
inquiry                inconsistent   acct_048     inv_048_2
inquiry                domain_expert  acct_018     inv_018_3
void_invoice           rushed         acct_015     inv_015_3
void_invoice           anxious        acct_014     inv_014_2
void_invoice           rule_breaker   acct_008     inv_008_4
void_invoice           inconsistent   acct_047     inv_047_1
void_invoice           domain_expert  acct_jordan  inv_jordan_4
refuse_void            rushed         acct_045     inv_045_2
refuse_void            anxious        acct_036     inv_036_3
refuse_void            rule_breaker   acct_007     inv_007_4
refuse_void            inconsistent   acct_039     inv_039_4
refuse_void            domain_expert  acct_029     inv_029_2
update_billing_email   rushed         acct_maya    inv_maya_4

## Step 4 — Verify and save

Apply domain rules, exact oracle FSM validation, replay, and action-sequence deduplication. Only passing tasks are persisted.

In [6]:
from aieng.syn_data.synbench.generation.generator import GenerationRun


run = GenerationRun(domain, seed=42)
verified_tasks, rejections = run.verify_drafts(drafts)
task_path = run.write_tasks(verified_tasks, OUT_DIR)

print("Drafts by type:", dict(Counter(task.task_type for task in drafts)))
print("Passing by type:", dict(Counter(task.task_type for task in verified_tasks)))
print("Rejected by type:", dict(Counter(task.task_type for task, _ in rejections)))
print("Saved", len(verified_tasks), "tasks to", task_path)

for task, reasons in rejections:
    print(f"REJECTED {task.id}: {'; '.join(reasons)}")

Drafts by type: {'inquiry': 5, 'void_invoice': 5, 'refuse_void': 5, 'update_billing_email': 5, 'update_seats': 5}
Passing by type: {'inquiry': 5, 'void_invoice': 5, 'refuse_void': 5, 'update_billing_email': 5, 'update_seats': 5}
Rejected by type: {}
Saved 25 tasks to /Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/agent_benchmark_generation/data/benchmarks/mock_saas_billing/tasks.json


## Step 5 — Load and re-verify saved tasks

Reload the persisted JSON through the task schema and independently verify every task before evaluation.


**Note: don't forget to load and inspect the domain before running this cell**

In [4]:
from aieng.syn_data.synbench.schemas.tasks import Task
from aieng.syn_data.synbench.verification.pipeline import verify_draft


TASK_PATH = OUT_DIR / "tasks.json"

with TASK_PATH.open() as f:
    saved = json.load(f)


filtered_tasks = []
for item in saved["tasks"]:
    task = Task.model_validate(item)
    verification = verify_draft(domain, task)
    status = "OK" if verification.verification_report.passed else "FAIL"
    print(f"[{status}] {task.id}")
    if verification.verification_report.passed:
        filtered_tasks.append(task)

print(f"Ready to evaluate: {len(filtered_tasks)} tasks")

[OK] scale_inquiry_rushed
[OK] scale_inquiry_anxious
[OK] scale_inquiry_rule_breaker
[OK] scale_inquiry_inconsistent
[OK] scale_inquiry_domain_expert
[OK] scale_void_invoice_rushed
[OK] scale_void_invoice_anxious
[OK] scale_void_invoice_rule_breaker
[OK] scale_void_invoice_inconsistent
[OK] scale_void_invoice_domain_expert
[OK] scale_refuse_void_rushed
[OK] scale_refuse_void_anxious
[OK] scale_refuse_void_rule_breaker
[OK] scale_refuse_void_inconsistent
[OK] scale_refuse_void_domain_expert
[OK] scale_update_billing_email_rushed
[OK] scale_update_billing_email_anxious
[OK] scale_update_billing_email_rule_breaker
[OK] scale_update_billing_email_inconsistent
[OK] scale_update_billing_email_domain_expert
[OK] scale_update_seats_rushed
[OK] scale_update_seats_anxious
[OK] scale_update_seats_rule_breaker
[OK] scale_update_seats_inconsistent
[OK] scale_update_seats_domain_expert
Ready to evaluate: 25 tasks


## Step 6 — Evaluate the multi-agent system

Use the same user simulator → planner → executor → critic pipeline as notebook 4. First inspect one score, then compute pass@1 and mean rewards over all validated tasks.


First inspect the agentic system on one task. Is it evaluated correctly? How can you improve the evaluation?



### Inspect an inquiry task

## Evaluate a sampled `void_invoice` tasks
Let's evaluate two agentic systems that use different LLMs

- Agent 1 LLM: `gpt-40`

- Agent 2 LLM: `gemini-3-flash-preview` A stronger but more expensive model

First, load the task:

In [14]:
TASK_PATH = OUT_DIR / "example_void_invoice_task.json"

with TASK_PATH.open() as f:
    saved = json.load(f)


example_task = Task.model_validate(saved["tasks"][0])
verification = verify_draft(domain, example_task)
status = "OK" if verification.verification_report.passed else "FAIL"
print(f"[{status}] {example_task.id}")
assert verification.verification_report.passed

show_task(example_task, title=" Example Void Invoice Task")

[OK] scale_void_invoice_rushed


###  Example Void Invoice Task
**id:** `scale_void_invoice_rushed`  
**type:** `void_invoice`  
**description:** Void an open or past-due invoice

#### User scenario
- **user name:** Ben Cole
- **personality style:** rushed
- **instructions:** The user wants to void invoice inv_015_3. If asked for identity, provide name Ben Cole.
- **initial message:** Hey, I need to void that invoice right now. It's an error. Can you help?

#### Evaluation criteria
- **reward basis:** DB, COMMUNICATE
- **communicate info:** ['voided']
- **oracle actions:**
  - `get_invoice` `{"invoice_id": "inv_015_3"}`
  - `void_invoice` `{"invoice_id": "inv_015_3"}`

#### Agent 1 (`gpt-4o`) fails the task

This agentic systems fails to ask the user the invoice ID and voids the first .

Note that after a trace like planner → executor → critic → user_sim → planner → …, the table shows the plan from the last planner step. Earlier plans are gone from session.plan

In [15]:
from aieng.syn_data.synbench.agents.pipeline import DEFAULT_ROLES, AgentPipeline
from aieng.syn_data.synbench.llm import get_client
from aieng.syn_data.synbench.llm.config import get_agent_model, get_user_simulator_model


# Agent uses `gpt-4o`
agent_model = "gpt-4o"
agent_client = get_client(agent_model)
print(" Agent Model:", agent_model)
print("Role order:", DEFAULT_ROLES)

user_model = get_user_simulator_model()
user_client = get_client(user_model)
print(" User Model:", user_model)

pipeline = AgentPipeline(
    domain,
    max_turns=12,
    max_dialogue_turns=3,
    agent_client=agent_client,
    user_simulator_client=user_client,
)

sample_session = pipeline.run_task(example_task)
sample_score = pipeline.score_session(sample_session)
print("First task:", example_task.id)
print("Reward:", sample_score.reward)
print("DB reward:", sample_score.db_reward)
print("Communication reward:", sample_score.communicate_reward)
if sample_score.missing_communicate:
    print("Missing communicate:", sample_score.missing_communicate)

 Agent Model: gpt-4o
Role order: ['user_sim', 'planner', 'executor', 'critic']
 User Model: gpt-4o
First task: scale_void_invoice_rushed
Reward: 0.0
DB reward: 0.0
Communication reward: 1.0


In [16]:
# Inspect the session
show_session(sample_session, title="Session (after dialogue)", show_chat=False)
show_pipeline_messages(sample_session, title="Pipeline messages")
show_actions(sample_session.agent_actions, title="Agent actions")

### Session (after dialogue)

| Field | Value |
| --- | --- |
| **domain** | `mock_saas_billing` (`/Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/agent_benchmark_generation/domains/mock_saas_billing`) |
| **task** | `scale_void_invoice_rushed` · `void_invoice` |
| **env** | ready |
| **messages** | 15 |
| **agent_actions** | 4 |
| **agent_messages** | 3 |
| **role_trace** | ['planner', 'executor', 'critic', 'user_sim', 'planner', 'executor', 'critic', 'user_sim', 'planner', 'executor', 'critic'] |
| **plan** | 1. Ask for the customer's full name to locate the account and verify access to the correct invoices.
2. Use the provided full name with `find_account_id` to find the account ID.
3. Use `list_invoices` with the account ID to confirm the status and details of invoice `inv_015_3`.
4. If invoice `inv_015_3` has `open` or `past_due` status, proceed to void it.
5. Confirm the successful voiding of invoice `inv_015_3` to the customer and provide clarification if the invoice cannot be voided. |

### Agent actions
1. `find_account_id` — `{"name": "Ben Cole"}`
2. `list_invoices` — `{"account_id": "acct_015"}`
3. `void_invoice` — `{"invoice_id": "inv_015_1"}`
4. `void_invoice` — `{"invoice_id": "inv_015_3"}`

### Agent replies

> I can help with that. Could you please provide your full name so that I can locate your account and check the invoice status?

> The invoice with ID `inv_015_1`, which was in a `past_due` status, has been successfully voided. If you have any more questions or need further assistance, feel free to ask!

> The invoice with ID `inv_015_3`, which was in an `open` status, has now been successfully voided. If there's anything else you need help with, please let me know!

### Pipeline messages

**role_trace:** `planner → executor → critic → user_sim → planner → executor → critic → user_sim → planner → executor → critic`

**plan (planner):**
```
1. Ask for the customer's full name to locate the account and verify access to the correct invoices.
2. Use the provided full name with `find_account_id` to find the account ID.
3. Use `list_invoices` with the account ID to confirm the status and details of invoice `inv_015_3`.
4. If invoice `inv_015_3` has `open` or `past_due` status, proceed to void it.
5. Confirm the successful voiding of invoice `inv_015_3` to the customer and provide clarification if the invoice cannot be voided.
```

**[0] system**
    You are the billing support agent for this domain.
    task_id: scale_void_invoice_rushed

    ## Policy
    # SaaS Billing Support Policy

    You are a billing support agent for a SaaS product.

    ## Identity and ownership

    1. Customers do not know their `account_id`. Ask for the customer's full name, then use `find_account_id`. Never invent or guess an ID.
    2. If no account matches, ask the customer to confirm their full name. Do not continue with invoice or subscription tools.
    3. Use the retrieved account ID with `list_invoices` or `list_subscriptions` before answering account-level questions or making changes.
    4. Never access or modify an invoice or subscription belonging to another account.

    ## Invoices

    5. Look up an invoice and confirm its amount, status, and due date before
    treating an inquiry as complete or attempting to void it. 
    6. Only invoices with `open` or `past_due` status may be voided.
    7. A `paid` or already `void` invoice cannot be voided. Explain the status when refusing.
    8. Confirm successful voiding to the customer.

    ## Account and subscription changes

    9. Confirm a new billing email with the customer before updating it, and repeat the updated email in the final response.
    10. Look up subscriptions before changing seats. Seats may only be changed on an `active` subscription, and the new count must be a positive integer.
    11. Repeat the updated seat count in the final response.

    ## Communication

    - Be concise, professional, and explicit about completed or refused changes.
    - Do not expose internal IDs unless the customer provided the corresponding invoice identifier.
    - When reporting an invoice details or information, don't convert or change the format of dates.


    Resolve the customer's messages using the provided tools and policy.
    Do not invent goals beyond what the customer says.
    When finished, reply to the customer clearly.

    ## Plan from planner
    1. Ask for the customer's full name to locate the account and verify access to the correct invoices.
    2. Use the provided full name with `find_account_id` to find the account ID.
    3. Use `list_invoices` with the account ID to confirm the status and details of invoice `inv_015_3`.
    4. If invoice `inv_015_3` has `open` or `past_due` status, proceed to void it.
    5. Confirm the successful voiding of invoice `inv_015_3` to the customer and provide clarification if the invoice cannot be voided.

**[1] user**
    Hey, I need to void that invoice right now. It's an error. Can you help?

**[2] assistant (executor)**
    I can help with that. Could you please provide your full name so that I can locate your account and check the invoice status?

**[3] user**
    Ben Cole.

**[4] assistant (executor)**
    tool_call `find_account_id`
    {
      "name": "Ben Cole"
    }

**[5] tool**
    {
      "account_id": "acct_015",
      "name": "Ben Cole",
      "email": "ben.cole@example.com",
      "billing_email": "billing+ben.cole@example.com",
      "company": "Cole 15 Software",
      "plan": "business"
    }

**[6] assistant (executor)**
    tool_call `list_invoices`
    {
      "account_id": "acct_015"
    }

**[7] tool**
    [
      {
        "invoice_id": "inv_015_1",
        "account_id": "acct_015",
        "subscription_id": "sub_015_1",
        "status": "past_due",
        "amount_cents": 24882,
        "currency": "USD",
        "due_date": "2026-07-30",
        "billing_address": "114 Market St, Suite 1",
        "line_items": [
          {
            "description": "Business plan seats",
            "quantity": 26,
            "unit_cents": 957
          }
        ]
      },
      {
        "invoice_id": "inv_015_2",
        "account_id": "acct_015",
        "subscription_id": "sub_015_2",
        "status": "void",
        "amount_cents": 5400,
        "currency": "USD",
        "due_date": "2026-08-13",
        "billing_address": "114 Market St, Suite 2",
        "line_items": [
          {
            "description": "Starter plan seats",
            "quantity": 6,
            "unit_cents": 900
          }
        ]
      },
      {
        "invoice_id": "inv_015_3",
        "account_id": "acct_015",
        "subscription_id": "sub_015_1",
        "status": "open",
        "amount_cents": 24882,
        "currency": "USD",
        "due_date": "2026-08-27",
        "billing_address": "114 Market St, Suite 3",
        "line_items": [
          {
            "description": "Business plan seats",
            "quantity": 26,
            "unit_cents": 957
          }
        ]
      },
      {
        "invoice_id": "inv_015_4",
        "account_id": "acct_015",
        "subscription_id": "sub_015_2",
        "status": "paid",
        "amount_cents": 5400,
        "currency": "USD",
        "due_date": "2026-09-10",
        "billing_address": "114 Market St, Suite 4",
        "line_items": [
          {
            "description": "Starter plan seats",
            "quantity": 6,
            "unit_cents": 900
          }
        ]
      }
    ]

**[8] assistant (executor)**
    tool_call `void_invoice`
    {
      "invoice_id": "inv_015_1"
    }

**[9] tool**
    {
      "invoice_id": "inv_015_1",
      "status": "void"
    }

**[10] assistant (executor)**
    The invoice with ID `inv_015_1`, which was in a `past_due` status, has been successfully voided. If you have any more questions or need further assistance, feel free to ask!

**[11] user**
    I needed to void invoice inv_015_3, not inv_015_1. Can you fix that?

**[12] assistant (executor)**
    tool_call `void_invoice`
    {
      "invoice_id": "inv_015_3"
    }

**[13] tool**
    {
      "invoice_id": "inv_015_3",
      "status": "void"
    }

**[14] assistant (executor)**
    The invoice with ID `inv_015_3`, which was in an `open` status, has now been successfully voided. If there's anything else you need help with, please let me know!


### Agent actions
1. `find_account_id` — `{"name": "Ben Cole"}`
2. `list_invoices` — `{"account_id": "acct_015"}`
3. `void_invoice` — `{"invoice_id": "inv_015_1"}`
4. `void_invoice` — `{"invoice_id": "inv_015_3"}`

### Use a stronger agent (the default agent)

Our agentic pipeline used `gemini-3-flash-preview` by default.

In [ ]:
from aieng.syn_data.synbench.agents.pipeline import DEFAULT_ROLES, AgentPipeline
from aieng.syn_data.synbench.llm import get_client
from aieng.syn_data.synbench.llm.config import get_user_simulator_model


# Agent uses `gemini-3-flash-preview`
agent_model = get_agent_model()
agent_client = get_client(agent_model)
print(" Agent Model:", agent_model)
print("Role order:", DEFAULT_ROLES)

# User simulator uses `gpt-4o`
user_model = get_user_simulator_model()
user_client = get_client(user_model)
print(" User Model:", user_model)

pipeline = AgentPipeline(
    domain,
    max_turns=12,
    max_dialogue_turns=3,
    agent_client=agent_client,
    user_simulator_client=user_client,
)

sample_session = pipeline.run_task(example_task)
sample_score = pipeline.score_session(sample_session)
print("First task:", example_task.id)
print("Reward:", sample_score.reward)
print("DB reward:", sample_score.db_reward)
print("Communication reward:", sample_score.communicate_reward)
if sample_score.missing_communicate:
    print("Missing communicate:", sample_score.missing_communicate)

 Agent Model: gemini-3-flash-preview
Role order: ['user_sim', 'planner', 'executor', 'critic']
 User Model: gpt-4o
First task: scale_void_invoice_rushed
Reward: 1.0
DB reward: 1.0
Communication reward: 1.0



A strong agent knows that it should confirm the invoice with the customer before changing it. 

Agent message to the customer:

```markdown
I found two invoices that can be voided on your account:

1.  **$248.82** (Status: `past_due`, Due Date: **2026-07-30**)
2.  **$248.82** (Status: `open`, Due Date: **2026-08-27**)

Could you please confirm which invoice you would like me to void?

```

In [ ]:
# Inspect the messages
show_pipeline_messages(sample_session, title="Weaker agent pipeline messages")

### Weaker agent pipeline messages

**role_trace:** `planner → executor → critic → user_sim → planner → executor → critic → user_sim → planner → executor → critic`

**plan (planner):**
```
1. Use the invoice ID provided by the customer to retrieve and confirm the details (amount, status, and due date) of invoice inv_015_3.
2. Verify that the invoice status is either "open" or "past_due" to ensure it is eligible for voiding.
3. Void invoice inv_015_3 using the appropriate tool.
4. Confirm to the customer that the invoice has been successfully voided.
```

**[0] system**
    You are the billing support agent for this domain.
    task_id: scale_void_invoice_rushed

    ## Policy
    # SaaS Billing Support Policy

    You are a billing support agent for a SaaS product.

    ## Identity and ownership

    1. Customers do not know their `account_id`. Ask for the customer's full name, then use `find_account_id`. Never invent or guess an ID.
    2. If no account matches, ask the customer to confirm their full name. Do not continue with invoice or subscription tools.
    3. Use the retrieved account ID with `list_invoices` or `list_subscriptions` before answering account-level questions or making changes.
    4. Never access or modify an invoice or subscription belonging to another account.

    ## Invoices

    5. Look up an invoice and confirm its amount, status, and due date before
    treating an inquiry as complete or attempting to void it. 
    6. Only invoices with `open` or `past_due` status may be voided.
    7. A `paid` or already `void` invoice cannot be voided. Explain the status when refusing.
    8. Confirm successful voiding to the customer.

    ## Account and subscription changes

    9. Confirm a new billing email with the customer before updating it, and repeat the updated email in the final response.
    10. Look up subscriptions before changing seats. Seats may only be changed on an `active` subscription, and the new count must be a positive integer.
    11. Repeat the updated seat count in the final response.

    ## Communication

    - Be concise, professional, and explicit about completed or refused changes.
    - Do not expose internal IDs unless the customer provided the corresponding invoice identifier.
    - When reporting an invoice details or information, don't convert or change the format of dates.


    Resolve the customer's messages using the provided tools and policy.
    Do not invent goals beyond what the customer says.
    When finished, reply to the customer clearly.

    ## Plan from planner
    1. Use the invoice ID provided by the customer to retrieve and confirm the details (amount, status, and due date) of invoice inv_015_3.
    2. Verify that the invoice status is either "open" or "past_due" to ensure it is eligible for voiding.
    3. Void invoice inv_015_3 using the appropriate tool.
    4. Confirm to the customer that the invoice has been successfully voided.

**[1] user**
    Hey, I need to void that invoice right now. It's an error. Can you help?

**[2] assistant (executor)**
    I'd be happy to help you with that. To get started, could you please provide your full name so I can locate your account?

**[3] user**
    Ben Cole.

**[4] assistant (executor)**
    tool_call `find_account_id`
    {
      "name": "Ben Cole"
    }

**[5] tool**
    {
      "account_id": "acct_015",
      "name": "Ben Cole",
      "email": "ben.cole@example.com",
      "billing_email": "billing+ben.cole@example.com",
      "company": "Cole 15 Software",
      "plan": "business"
    }

**[6] assistant (executor)**
    tool_call `list_invoices`
    {
      "account_id": "acct_015"
    }

**[7] tool**
    [
      {
        "invoice_id": "inv_015_1",
        "account_id": "acct_015",
        "subscription_id": "sub_015_1",
        "status": "past_due",
        "amount_cents": 24882,
        "currency": "USD",
        "due_date": "2026-07-30",
        "billing_address": "114 Market St, Suite 1",
        "line_items": [
          {
            "description": "Business plan seats",
            "quantity": 26,
            "unit_cents": 957
          }
        ]
      },
      {
        "invoice_id": "inv_015_2",
        "account_id": "acct_015",
        "subscription_id": "sub_015_2",
        "status": "void",
        "amount_cents": 5400,
        "currency": "USD",
        "due_date": "2026-08-13",
        "billing_address": "114 Market St, Suite 2",
        "line_items": [
          {
            "description": "Starter plan seats",
            "quantity": 6,
            "unit_cents": 900
          }
        ]
      },
      {
        "invoice_id": "inv_015_3",
        "account_id": "acct_015",
        "subscription_id": "sub_015_1",
        "status": "open",
        "amount_cents": 24882,
        "currency": "USD",
        "due_date": "2026-08-27",
        "billing_address": "114 Market St, Suite 3",
        "line_items": [
          {
            "description": "Business plan seats",
            "quantity": 26,
            "unit_cents": 957
          }
        ]
      },
      {
        "invoice_id": "inv_015_4",
        "account_id": "acct_015",
        "subscription_id": "sub_015_2",
        "status": "paid",
        "amount_cents": 5400,
        "currency": "USD",
        "due_date": "2026-09-10",
        "billing_address": "114 Market St, Suite 4",
        "line_items": [
          {
            "description": "Starter plan seats",
            "quantity": 6,
            "unit_cents": 900
          }
        ]
      }
    ]

**[8] assistant (executor)**
    I've found two invoices that can be voided for your account. Which one would you like me to void?

    1. **Amount: $248.82**, Due Date: **2026-07-30** (Status: past_due)
    2. **Amount: $248.82**, Due Date: **2026-08-27** (Status: open)

**[9] user**
    It's inv_015_3 I need voided.

**[10] assistant (executor)**
    tool_call `void_invoice`
    {
      "invoice_id": "inv_015_3"
    }

**[11] tool**
    {
      "invoice_id": "inv_015_3",
      "status": "void"
    }

**[12] assistant (executor)**
    I have successfully voided invoice inv_015_3.

    **Invoice Details:**
    - **Amount:** $248.82
    - **Due Date:** 2026-08-27
    - **Status:** void (Updated from open)


# Evaluate all the tasks

This can spend your API credits, so be careful about running agentic pipeline on large generated benchmarks.


 We use `pass@1` for the metric because we want to measure if the agent can complete the task in the first trial Successfully, but this can introduce some randomness as the agent can do the task successfully for the first time and fail the next time.

For a more trustworthy evaluation, you can run the set of tasks several times. Such metric is introduced in tau-bench as `pass^k` (different from `pass@k`), which measures the agent's **reliability** and determines if it can successfully complete the same task **multiple** times (`k` representing the number of different trials).

In [5]:
from aieng.syn_data.synbench.agents.pipeline import DEFAULT_ROLES, AgentPipeline
from aieng.syn_data.synbench.evaluation.metrics import MetricsCollector
from aieng.syn_data.synbench.llm import get_client
from aieng.syn_data.synbench.llm.config import get_agent_model, get_user_simulator_model


RESULT_DIR = BENCHMARK_RESULTS_DIR / "gemini-3-flash-preview"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
# Agent uses `gemini-3-flash-preview``
agent_model = get_agent_model()
agent_client = get_client(agent_model)
print(" Agent Model:", agent_model)
print("Role order:", DEFAULT_ROLES)

# User simulator uses `gpt-4o`
user_model = get_user_simulator_model()
user_client = get_client(user_model)
print(" User Model:", user_model)

pipeline = AgentPipeline(
    domain,
    max_turns=12,  # tool-call rounds per user utterance
    max_dialogue_turns=5,  # user ↔ pipeline rounds
    agent_client=agent_client,
    user_simulator_client=user_client,
)

gemini_flash_metrics = MetricsCollector()
for task in filtered_tasks:
    score = pipeline.run_and_score_task(task)
    gemini_flash_metrics.add(task.id, score)
    print(
        f"{task.id}: reward={score.reward:.2f} dialogue_turns={score.dialogue_turns}",
    )

gemini_flash_summary = gemini_flash_metrics.summary()

with open(RESULT_DIR / "benchmark_summary.json", "w") as f:
    json.dump(gemini_flash_summary, f, indent=2)

print("\n Saved benchmark summary at", RESULT_DIR / "benchmark_summary.json")

 Agent Model: gemini-3-flash-preview
Role order: ['user_sim', 'planner', 'executor', 'critic']
 User Model: gpt-4o
scale_inquiry_rushed: reward=1.00 dialogue_turns=3
scale_inquiry_anxious: reward=1.00 dialogue_turns=5
scale_inquiry_rule_breaker: reward=0.00 dialogue_turns=2
scale_inquiry_inconsistent: reward=1.00 dialogue_turns=4
scale_inquiry_domain_expert: reward=0.00 dialogue_turns=4
scale_void_invoice_rushed: reward=1.00 dialogue_turns=3
scale_void_invoice_anxious: reward=1.00 dialogue_turns=3
scale_void_invoice_rule_breaker: reward=1.00 dialogue_turns=3
scale_void_invoice_inconsistent: reward=1.00 dialogue_turns=4
scale_void_invoice_domain_expert: reward=1.00 dialogue_turns=3
scale_refuse_void_rushed: reward=0.00 dialogue_turns=5
scale_refuse_void_anxious: reward=0.00 dialogue_turns=4
scale_refuse_void_rule_breaker: reward=1.00 dialogue_turns=5
scale_refuse_void_inconsistent: reward=0.00 dialogue_turns=3
scale_refuse_void_domain_expert: reward=0.00 dialogue_turns=2
scale_update_bi

In [7]:
# A summary of the benchmark results
show_benchmark_by_task_type(
    gemini_flash_metrics,
    filtered_tasks,
    agent_model=agent_model,
    task_types=list(domain.task_types.keys()),
)

with open(RESULT_DIR / "benchmark_summary.json", "r") as f:
    summary = json.load(f)

# print the summary
print("n_tasks:", summary["n_tasks"])
print("pass_at_1:", summary["pass_at_1"])
print("mean_db_reward:", summary["mean_db_reward"])
print("mean_communicate_reward:", summary["mean_communicate_reward"])
print("mean_dialogue_turns:", summary["mean_dialogue_turns"])

**Agent LLM:** `gemini-3-flash-preview`

| task_type | rushed | anxious | rule_breaker | inconsistent | domain_expert | n_pass | n | pass_rate | avg_turns |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| `inquiry` | 1 | 1 | 0 | 1 | 0 | 3 | 5 | 0.60 | 3.60 |
| `void_invoice` | 1 | 1 | 1 | 1 | 1 | 5 | 5 | 1.00 | 3.20 |
| `refuse_void` | 0 | 0 | 1 | 0 | 0 | 1 | 5 | 0.20 | 3.80 |
| `update_billing_email` | 1 | 1 | 1 | 1 | 1 | 5 | 5 | 1.00 | 4.40 |
| `update_seats` | 1 | 1 | 1 | 1 | 1 | 5 | 5 | 1.00 | 3.60 |

n_tasks: 25
pass_at_1: 0.76
mean_db_reward: 0.92
mean_communicate_reward: 0.76
mean_dialogue_turns: 3.72


### Analyze a potentially hard task
See what happens if you run one of the failed tasks again. Will it fail again or can it pass?



In [9]:
from aieng.syn_data.synbench.agents.pipeline import DEFAULT_ROLES, AgentPipeline
from aieng.syn_data.synbench.llm import get_client
from aieng.syn_data.synbench.llm.config import get_agent_model, get_user_simulator_model


# Retrieve the hard task that scored 0.00 in the full benchmark run
hard_task_id = "scale_refuse_void_domain_expert"
hard_task = next(task for task in filtered_tasks if task.id == hard_task_id)
show_task(hard_task, title=f"Hard task: {hard_task_id}")

# Inspect the task and run results with the default agent LLM
# Agent uses `gemini-3-flash-preview` by default
agent_model = get_agent_model()
agent_client = get_client(agent_model)
print(" Agent Model:", agent_model)
print("Role order:", DEFAULT_ROLES)

user_model = get_user_simulator_model()
user_client = get_client(user_model)
print(" User Model:", user_model)

pipeline = AgentPipeline(
    domain,
    max_turns=12,
    max_dialogue_turns=5,
    agent_client=agent_client,
)

hard_session = pipeline.run_task(hard_task)
hard_case_score = pipeline.score_session(hard_session)
print("First task:", hard_task.id)
print("Reward:", hard_case_score.reward)
print("DB reward:", hard_case_score.db_reward)
print("Communication reward:", hard_case_score.communicate_reward)
if hard_case_score.missing_communicate:
    print("Missing communicate:", hard_case_score.missing_communicate)

show_session(hard_session, title="Session (stronger agent)", show_chat=False)
show_pipeline_messages(hard_session, title="Pipeline messages")
show_actions(hard_session.agent_actions, title="Agent actions")

### Hard task: scale_refuse_void_domain_expert
**id:** `scale_refuse_void_domain_expert`  
**type:** `refuse_void`  
**description:** Refuse to void a paid invoice

#### User scenario
- **user name:** Diego Evans
- **personality style:** domain_expert
- **instructions:** Request to void the paid invoice inv_029_2. Accept the policy explanation that paid invoices cannot be voided.
- **initial message:** Hello. I need to initiate a void operation on one of our recent invoices. I am aware of the standard protocol, so please let me know when you are ready for my full name and the specific invoice identifier to perform the necessary ownership and status checks.

#### Evaluation criteria
- **reward basis:** DB, COMMUNICATE
- **communicate info:** ['cannot be voided']
- **oracle actions:**
  - `get_invoice` `{"invoice_id": "inv_029_2"}`

 Agent Model: gemini-3-flash-preview
Role order: ['user_sim', 'planner', 'executor', 'critic']
 User Model: gpt-4o
First task: scale_refuse_void_domain_expert
Reward: 0.0
DB reward: 1.0
Communication reward: 0.0
Missing communicate: ['cannot be voided']


### Session (stronger agent)

| Field | Value |
| --- | --- |
| **domain** | `mock_saas_billing` (`/Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/agent_benchmark_generation/domains/mock_saas_billing`) |
| **task** | `scale_refuse_void_domain_expert` · `refuse_void` |
| **env** | ready |
| **messages** | 9 |
| **agent_actions** | 2 |
| **agent_messages** | 2 |
| **role_trace** | ['planner', 'executor', 'critic', 'user_sim', 'planner', 'executor', 'critic', 'critic', 'user_sim'] |
| **plan** | 1. Use `find_account_id` with the name "Diego Evans" to retrieve the customer's account ID.
2. Use `list_invoices` for the retrieved account ID and identify invoice `inv_029_2`.
3. Confirm the invoice's amount, status, and due date.
4. If the status is `open` or `past_due`, use `void_invoice` and confirm the action to the customer; otherwise, refuse and explain based on the current status. |

### Agent actions
1. `find_account_id` — `{"name": "Diego Evans"}`
2. `list_invoices` — `{"account_id": "acct_029"}`

### Agent replies

> I am ready to help you with that. Please provide your full name and the invoice identifier so I can verify your account and the status of the invoice.

> I have reviewed the details for invoice inv_029_2. This invoice for $248.82 was due on 2027-02-25 and its current status is "paid".
> 
> Per our policy, only invoices with an "open" or "past_due" status can be voided. Because this invoice has already been paid, I am unable to initiate a void operation for it.

### Pipeline messages

**role_trace:** `planner → executor → critic → user_sim → planner → executor → critic → critic → user_sim`

**plan (planner):**
```
1. Use `find_account_id` with the name "Diego Evans" to retrieve the customer's account ID.
2. Use `list_invoices` for the retrieved account ID and identify invoice `inv_029_2`.
3. Confirm the invoice's amount, status, and due date.
4. If the status is `open` or `past_due`, use `void_invoice` and confirm the action to the customer; otherwise, refuse and explain based on the current status.
```

**[0] system**
    You are the billing support agent for this domain.
    task_id: scale_refuse_void_domain_expert

    ## Policy
    # SaaS Billing Support Policy

    You are a billing support agent for a SaaS product.

    ## Identity and ownership

    1. Customers do not know their `account_id`. Ask for the customer's full name, then use `find_account_id`. Never invent or guess an ID.
    2. If no account matches, ask the customer to confirm their full name. Do not continue with invoice or subscription tools.
    3. Use the retrieved account ID with `list_invoices` or `list_subscriptions` before answering account-level questions or making changes.
    4. Never access or modify an invoice or subscription belonging to another account.

    ## Invoices

    5. Look up an invoice and confirm its amount, status, and due date before
    treating an inquiry as complete or attempting to void it. 
    6. Only invoices with `open` or `past_due` status may be voided.
    7. A `paid` or already `void` invoice cannot be voided. Explain the status when refusing.
    8. Confirm successful voiding to the customer.

    ## Account and subscription changes

    9. Confirm a new billing email with the customer before updating it, and repeat the updated email in the final response.
    10. Look up subscriptions before changing seats. Seats may only be changed on an `active` subscription, and the new count must be a positive integer.
    11. Repeat the updated seat count in the final response.

    ## Communication

    - Be concise, professional, and explicit about completed or refused changes.
    - Do not expose internal IDs unless the customer provided the corresponding invoice identifier.
    - When reporting an invoice details or information, don't convert or change the format of dates.


    Resolve the customer's messages using the provided tools and policy.
    Do not invent goals beyond what the customer says.
    When finished, reply to the customer clearly.

    ## Plan from planner
    1. Use `find_account_id` with the name "Diego Evans" to retrieve the customer's account ID.
    2. Use `list_invoices` for the retrieved account ID and identify invoice `inv_029_2`.
    3. Confirm the invoice's amount, status, and due date.
    4. If the status is `open` or `past_due`, use `void_invoice` and confirm the action to the customer; otherwise, refuse and explain based on the current status.

    Critic revision:
    Remove the account ID (acct_029) from the response, as policy states customers do not know their account ID and internal IDs should not be exposed unless provided by the customer.

**[1] user**
    Hello. I need to initiate a void operation on one of our recent invoices. I am aware of the standard protocol, so please let me know when you are ready for my full name and the specific invoice identifier to perform the necessary ownership and status checks.

**[2] assistant (executor)**
    I am ready to help you with that. Please provide your full name and the invoice identifier so I can verify your account and the status of the invoice.

**[3] user**
    Certainly. My full name is Diego Evans, and the invoice identifier in question is inv_029_2. Please proceed with the verification process.

**[4] assistant (executor)**
    tool_call `find_account_id`
    {
      "name": "Diego Evans"
    }

**[5] tool**
    {
      "account_id": "acct_029",
      "name": "Diego Evans",
      "email": "diego.evans@example.com",
      "billing_email": "billing+diego.evans@example.com",
      "company": "Evans 29 Software",
      "plan": "pro"
    }

**[6] assistant (executor)**
    tool_call `list_invoices`
    {
      "account_id": "acct_029"
    }

**[7] tool**
    [
      {
        "invoice_id": "inv_029_1",
        "account_id": "acct_029",
        "subscription_id": "sub_029_1",
        "status": "open",
        "amount_cents": 9900,
        "currency": "USD",
        "due_date": "2027-02-11",
        "billing_address": "128 Market St, Suite 1",
        "line_items": [
          {
            "description": "Pro plan seats",
            "quantity": 10,
            "unit_cents": 990
          }
        ]
      },
      {
        "invoice_id": "inv_029_2",
        "account_id": "acct_029",
        "subscription_id": "sub_029_2",
        "status": "paid",
        "amount_cents": 24882,
        "currency": "USD",
        "due_date": "2027-02-25",
        "billing_address": "128 Market St, Suite 2",
        "line_items": [
          {
            "description": "Business plan seats",
            "quantity": 26,
            "unit_cents": 957
          }
        ]
      },
      {
        "invoice_id": "inv_029_3",
        "account_id": "acct_029",
        "subscription_id": "sub_029_1",
        "status": "past_due",
        "amount_cents": 9900,
        "currency": "USD",
        "due_date": "2027-03-11",
        "billing_address": "128 Market St, Suite 3",
        "line_items": [
          {
            "description": "Pro plan seats",
            "quantity": 10,
            "unit_cents": 990
          }
        ]
      },
      {
        "invoice_id": "inv_029_4",
        "account_id": "acct_029",
        "subscription_id": "sub_029_2",
        "status": "void",
        "amount_cents": 24882,
        "currency": "USD",
        "due_date": "2027-03-25",
        "billing_address": "128 Market St, Suite 4",
        "line_items": [
          {
            "description": "Business plan seats",
            "quantity": 26,
            "unit_cents": 957
          }
        ]
      }
    ]

**[8] assistant (executor)**
    I have reviewed the details for invoice inv_029_2. This invoice for $248.82 was due on 2027-02-25 and its current status is "paid".

    Per our policy, only invoices with an "open" or "past_due" status can be voided. Because this invoice has already been paid, I am unable to initiate a void operation for it.


### Agent actions
1. `find_account_id` — `{"name": "Diego Evans"}`
2. `list_invoices` — `{"account_id": "acct_029"}`

## Run the benchmark on another agent

Change the default agent LLM to "gpt-4o".

In [ ]:
# Evaluate another agent

RESULT_DIR = BENCHMARK_RESULTS_DIR / "gpt-4o"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

agent_model = "gpt-4o"
agent_client = get_client(agent_model)
print(" Agent Model:", agent_model)
print("Role order:", DEFAULT_ROLES)

user_model = get_user_simulator_model()
user_client = get_client(user_model)
print(" User Model:", user_model)

pipeline = AgentPipeline(
    domain,
    max_turns=12,  # tool-call rounds per user utterance
    max_dialogue_turns=5,  # user ↔ pipeline rounds
    agent_client=agent_client,
    user_simulator_client=user_client,
)

gpt_metrics = MetricsCollector()
for task in filtered_tasks:
    score = pipeline.run_and_score_task(task)
    gpt_metrics.add(task.id, score)
    print(
        f"{task.id}: reward={score.reward:.2f} dialogue_turns={score.dialogue_turns}",
    )

gpt_summary = gpt_metrics.summary()
with open(RESULT_DIR / "benchmark_summary.json", "w") as f:
    json.dump(gemini_flash_summary, f, indent=2)

print("\n Saved benchmark summary at", RESULT_DIR / "benchmark_summary.json")

 Agent Model: gpt-4o
Role order: ['user_sim', 'planner', 'executor', 'critic']
 User Model: gpt-4o
scale_inquiry_rushed: reward=1.00 dialogue_turns=3
scale_inquiry_anxious: reward=1.00 dialogue_turns=4
scale_inquiry_rule_breaker: reward=1.00 dialogue_turns=5
scale_inquiry_inconsistent: reward=0.00 dialogue_turns=4
scale_inquiry_domain_expert: reward=1.00 dialogue_turns=3
scale_void_invoice_rushed: reward=1.00 dialogue_turns=2
scale_void_invoice_anxious: reward=1.00 dialogue_turns=3
scale_void_invoice_rule_breaker: reward=1.00 dialogue_turns=2
scale_void_invoice_inconsistent: reward=1.00 dialogue_turns=4
scale_void_invoice_domain_expert: reward=0.00 dialogue_turns=2
scale_refuse_void_rushed: reward=0.00 dialogue_turns=5
scale_refuse_void_anxious: reward=0.00 dialogue_turns=4
scale_refuse_void_rule_breaker: reward=1.00 dialogue_turns=5
scale_refuse_void_inconsistent: reward=0.00 dialogue_turns=4
scale_refuse_void_domain_expert: reward=1.00 dialogue_turns=2
scale_update_billing_email_rush

In [12]:
# A summary of the benchmark results

RESULT_DIR = BENCHMARK_RESULTS_DIR / "gpt-4o"
show_benchmark_by_task_type(
    gpt_metrics,
    filtered_tasks,
    agent_model=agent_model,
    task_types=list(domain.task_types.keys()),
)

with open(RESULT_DIR / "benchmark_summary.json", "r") as f:
    summary = json.load(f)

# print the summary
print("n_tasks:", summary["n_tasks"])
print("pass_at_1:", summary["pass_at_1"])
print("mean_db_reward:", summary["mean_db_reward"])
print("mean_communicate_reward:", summary["mean_communicate_reward"])
print("mean_dialogue_turns:", summary["mean_dialogue_turns"])

**Agent LLM:** `gpt-4o`

| task_type | rushed | anxious | rule_breaker | inconsistent | domain_expert | n_pass | n | pass_rate | avg_turns |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| `inquiry` | 1 | 1 | 1 | 0 | 1 | 4 | 5 | 0.80 | 3.80 |
| `void_invoice` | 1 | 1 | 1 | 1 | 0 | 4 | 5 | 0.80 | 2.60 |
| `refuse_void` | 0 | 0 | 1 | 0 | 1 | 2 | 5 | 0.40 | 4.00 |
| `update_billing_email` | 1 | 1 | 1 | 1 | 1 | 5 | 5 | 1.00 | 3.40 |
| `update_seats` | 1 | 1 | 1 | 1 | 0 | 4 | 5 | 0.80 | 3.00 |

n_tasks: 25
pass_at_1: 0.76
mean_db_reward: 0.92
mean_communicate_reward: 0.76
mean_dialogue_turns: 3.72
